# B1: Permit Lifecycle Tracking

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Understand the housing permit lifecycle** from application to occupancy
2. **Query timeline data** from the permit_events table
3. **Calculate pipeline metrics** (processing times, bottlenecks)
4. **Identify stalled projects** that need attention
5. **Visualize project timelines** and completion rates

## Why This Matters

A housing project takes **3-5 years** from initial application to people moving in. Understanding this pipeline helps:
- Identify bottlenecks (where projects get stuck)
- Predict completion dates
- Hold the city accountable for processing delays
- Inform policy decisions about streamlining

## The Housing Pipeline

```
Application → Zoning Review → Building Permit → Construction → Occupancy
   (Day 0)    (6-18 months)   (2-4 months)    (12-36 months)  (Complete!)
```

## Data Sources

This notebook uses:
- **projects** table - 115 projects with milestone dates
- **project_permits** table - 47 permits linked to projects  
- **permit_events** table - 58 individual events with timestamps
- **project_velocity** view - Calculated processing times

---

## NEW: Dual Timestamp System

**Critical data quality insight:** We track TWO dates for every event:

| Column | Meaning | Example |
|--------|---------|---------|
| `event_date` | When the action actually occurred | "2024-03-15" (permit filed) |
| `imported_at` | When we recorded it in our database | "2025-02-22" (data collection) |

**Why this matters:**
- `event_date` tells you the real-world timeline
- `imported_at` tells you data freshness
- Gap between them = how far behind our data collection is

```sql
-- Find events we recorded late
SELECT address_display, event_date, imported_at,
       JULIANDAY(imported_at) - JULIANDAY(event_date) as days_lag
FROM permit_events pe
JOIN projects p ON pe.project_id = p.id
ORDER BY days_lag DESC;
```

---

## 1. Setup

In [ ]:
import sys
import sqlite3
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Find project root and setup environment
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config with resolved paths
import json
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve paths
DB_PATH = ROOT / CONFIG['paths']['database']
DATA_DIR = ROOT / CONFIG['paths']['data_dir']
OUTPUT_DIR = ROOT / CONFIG['paths']['output_dir']

print(f"Project root: {ROOT}")
print(f"Database: {DB_PATH}")
print(f"Database exists: {DB_PATH.exists()}")

In [ ]:
# Import project modules
from modules.data_loader import load_csv
from modules.timeline_calculator import (
    classify_project_status, 
    STATUS_ORDER,
    identify_stalled_projects
)

## 2. Load Project Data with Timeline Dates

Load from database to access the new date fields populated by A5_buildingeye_import.

In [ ]:
def load_projects_with_dates(db_path: Path) -> pd.DataFrame:
    """Load housing projects with timeline date fields."""
    conn = sqlite3.connect(db_path)
    
    df = pd.read_sql_query("""
        SELECT 
            id,
            address_display,
            apn,
            net_units,
            year,
            permits,
            status,
            project_size_category,
            latitude,
            longitude,
            -- Timeline dates
            first_filed_date,
            zoning_approved_date,
            building_permit_date,
            construction_start_date,
            co_issued_date,
            is_completed,
            last_status_date,
            total_days,
            date_source
        FROM housing_projects
        ORDER BY net_units DESC
    """, conn)
    
    conn.close()
    
    # Convert date columns
    date_cols = ['first_filed_date', 'zoning_approved_date', 'building_permit_date',
                 'construction_start_date', 'co_issued_date', 'last_status_date']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    
    return df

# Load data
df_projects = load_projects_with_dates(DB_PATH)
print(f"Loaded {len(df_projects)} projects")

# Check date coverage
date_cols = ['first_filed_date', 'zoning_approved_date', 'building_permit_date', 'co_issued_date']
print("\nDate field coverage:")
for col in date_cols:
    if col in df_projects.columns:
        count = df_projects[col].notna().sum()
        pct = 100 * count / len(df_projects)
        print(f"  {col}: {count} projects ({pct:.1f}%)")

In [ ]:
# Show sample projects with dates
display_cols = ['address_display', 'net_units', 'first_filed_date', 'zoning_approved_date', 
                'building_permit_date', 'co_issued_date', 'is_completed']
available_cols = [c for c in display_cols if c in df_projects.columns]

print("Sample projects with timeline data:")
display(df_projects[df_projects['first_filed_date'].notna()][available_cols].head(10))

## 3. Load Permit Events with Dual Timestamps

The `permit_events` table captures each step in the permit lifecycle.

**Key columns:**
- `stage` - Pipeline stage (Completeness Review, CEQA, Staff Decision, Appeal, etc.)
- `action` - What happened (Approved, Submitted, Appealed, etc.)
- `event_date` - When the action actually occurred
- `imported_at` - When we recorded this data
- `marked_by` - City staff who processed the action

In [ ]:
def load_permit_events(db_path: Path) -> pd.DataFrame:
    """
    Load all permit events with dual timestamps.
    
    Returns DataFrame with:
    - event_date: When the action occurred (real-world date)
    - imported_at: When we recorded the data (data collection date)
    """
    conn = sqlite3.connect(db_path)
    
    df = pd.read_sql_query("""
        SELECT 
            pe.id,
            pe.project_id,
            pe.permit_id,
            pe.permit_number,
            pe.permit_type,
            pe.stage,
            pe.action,
            pe.stage_status,
            pe.event_date,      -- When it actually happened
            pe.imported_at,     -- When we recorded it
            pe.marked_by,
            pe.notes,
            p.address_display,
            p.net_units
        FROM permit_events pe
        LEFT JOIN projects p ON pe.project_id = p.id
        ORDER BY pe.event_date DESC
    """, conn)
    
    conn.close()
    
    # Convert dates
    df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')
    df['imported_at'] = pd.to_datetime(df['imported_at'], errors='coerce')
    
    # Calculate data lag (days between event and when we recorded it)
    df['data_lag_days'] = (df['imported_at'] - df['event_date']).dt.days
    
    return df

# Load events
df_events = load_permit_events(DB_PATH)
print(f"Loaded {len(df_events)} permit events")

if len(df_events) > 0:
    print(f"\nStages tracked:")
    print(df_events['stage'].value_counts())
    
    print(f"\nActions recorded:")
    print(df_events['action'].value_counts())
    
    # Show data freshness
    avg_lag = df_events['data_lag_days'].mean()
    print(f"\nData freshness: avg {avg_lag:.0f} days between event and recording")
else:
    print("\nNo permit events found. Run manual Accela data collection workflow.")

## 3a. Project Velocity Analysis

Use the `project_velocity` view to analyze processing speed by stage.

**Key insight:** The same project may have multiple permits (zoning, building, demolition). 
Track them all to understand the full timeline.

In [ ]:
# Load project velocity view
def load_project_velocity(db_path: Path) -> pd.DataFrame:
    """Load velocity analysis from the project_velocity view."""
    conn = sqlite3.connect(db_path)
    
    # Check if view exists
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='view' AND name='project_velocity'")
    
    if cursor.fetchone():
        df = pd.read_sql_query("SELECT * FROM project_velocity", conn)
    else:
        # Fallback: calculate from permit_events
        df = pd.read_sql_query("""
            SELECT 
                p.address_display,
                p.net_units,
                COUNT(DISTINCT pe.stage) as stages_completed,
                MIN(pe.event_date) as first_event,
                MAX(pe.event_date) as last_event,
                CAST(JULIANDAY(MAX(pe.event_date)) - JULIANDAY(MIN(pe.event_date)) AS INTEGER) as total_days
            FROM projects p
            JOIN permit_events pe ON p.id = pe.project_id
            GROUP BY p.id
            ORDER BY p.net_units DESC
        """, conn)
    
    conn.close()
    return df

# Load velocity data
df_velocity = load_project_velocity(DB_PATH)
print(f"Projects with velocity data: {len(df_velocity)}")

if len(df_velocity) > 0:
    display(df_velocity.head(10))
else:
    print("No velocity data yet - collect more permit events from Accela.")

## 4. Calculate Timeline Metrics

Calculate days between each milestone.

In [ ]:
def calculate_timeline_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate timeline metrics for projects with date data.
    """
    df = df.copy()
    
    # Days from filing to zoning approval
    if 'first_filed_date' in df.columns and 'zoning_approved_date' in df.columns:
        df['days_to_zoning'] = (df['zoning_approved_date'] - df['first_filed_date']).dt.days
    
    # Days from zoning to building permit
    if 'zoning_approved_date' in df.columns and 'building_permit_date' in df.columns:
        df['days_zoning_to_building'] = (df['building_permit_date'] - df['zoning_approved_date']).dt.days
    
    # Days of construction (building permit to CO)
    if 'building_permit_date' in df.columns and 'co_issued_date' in df.columns:
        df['days_construction'] = (df['co_issued_date'] - df['building_permit_date']).dt.days
    
    # Total days (filing to CO)
    if 'first_filed_date' in df.columns and 'co_issued_date' in df.columns:
        df['days_total'] = (df['co_issued_date'] - df['first_filed_date']).dt.days
    
    # Days since last activity
    if 'last_status_date' in df.columns:
        today = pd.Timestamp.now()
        df['days_inactive'] = (today - df['last_status_date']).dt.days
    elif 'first_filed_date' in df.columns:
        today = pd.Timestamp.now()
        df['days_inactive'] = (today - df['first_filed_date']).dt.days
    
    return df

# Calculate metrics
df_timelines = calculate_timeline_metrics(df_projects)

# Show summary statistics
metric_cols = ['days_to_zoning', 'days_zoning_to_building', 'days_construction', 'days_total']
available_metrics = [c for c in metric_cols if c in df_timelines.columns and df_timelines[c].notna().any()]

if available_metrics:
    print("Timeline Statistics (days):")
    print("="*60)
    stats = df_timelines[available_metrics].describe().round(0)
    display(stats)
else:
    print("No timeline metrics available. Import date data first using A5_buildingeye_import.ipynb")

## 5. Pipeline Stage Analysis

Classify projects by their current stage in the pipeline.

In [ ]:
# Classify projects by status
if 'status' in df_timelines.columns:
    df_timelines['status_category'] = df_timelines['status'].apply(classify_project_status)

# Count by stage
print("Housing Development Pipeline:")
print("="*60)

stage_summary = []
for stage in STATUS_ORDER:
    if 'status_category' in df_timelines.columns:
        mask = df_timelines['status_category'] == stage
        count = mask.sum()
        units = df_timelines.loc[mask, 'net_units'].sum() if 'net_units' in df_timelines.columns else 0
    else:
        count = 0
        units = 0
    
    stage_summary.append({
        'Stage': stage.title(),
        'Projects': count,
        'Units': int(units)
    })
    print(f"{stage.upper():20} | {count:3} projects | {units:,} units")

df_stages = pd.DataFrame(stage_summary)

In [ ]:
# Visualize pipeline stages
if 'status_category' in df_timelines.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Projects by stage
    ax1 = axes[0]
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(df_stages)))
    bars1 = ax1.barh(df_stages['Stage'], df_stages['Projects'], color=colors)
    ax1.set_xlabel('Number of Projects')
    ax1.set_title('Projects by Pipeline Stage')
    ax1.invert_yaxis()
    for bar, val in zip(bars1, df_stages['Projects']):
        ax1.text(val + 0.5, bar.get_y() + bar.get_height()/2, str(val), va='center')
    
    # Units by stage
    ax2 = axes[1]
    colors = plt.cm.Greens(np.linspace(0.3, 0.9, len(df_stages)))
    bars2 = ax2.barh(df_stages['Stage'], df_stages['Units'], color=colors)
    ax2.set_xlabel('Number of Housing Units')
    ax2.set_title('Units by Pipeline Stage')
    ax2.invert_yaxis()
    for bar, val in zip(bars2, df_stages['Units']):
        ax2.text(val + 10, bar.get_y() + bar.get_height()/2, f"{val:,}", va='center')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'pipeline_stages.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {OUTPUT_DIR / 'pipeline_stages.png'}")

## 6. Timeline Distribution Analysis

How long does each stage take?

In [ ]:
# Plot timeline distributions
metric_cols = ['days_to_zoning', 'days_zoning_to_building', 'days_construction', 'days_total']
available_metrics = [c for c in metric_cols if c in df_timelines.columns and df_timelines[c].notna().sum() > 5]

if available_metrics:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    titles = {
        'days_to_zoning': 'Filing to Zoning Approval',
        'days_zoning_to_building': 'Zoning to Building Permit',
        'days_construction': 'Construction Duration',
        'days_total': 'Total Project Duration'
    }
    
    for i, col in enumerate(metric_cols):
        ax = axes[i]
        if col in available_metrics:
            data = df_timelines[col].dropna()
            ax.hist(data, bins=20, color='steelblue', edgecolor='white', alpha=0.7)
            ax.axvline(data.median(), color='red', linestyle='--', label=f'Median: {data.median():.0f} days')
            ax.axvline(data.mean(), color='orange', linestyle='--', label=f'Mean: {data.mean():.0f} days')
            ax.set_xlabel('Days')
            ax.set_ylabel('Number of Projects')
            ax.set_title(titles.get(col, col))
            ax.legend()
        else:
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(titles.get(col, col))
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'timeline_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {OUTPUT_DIR / 'timeline_distributions.png'}")
else:
    print("Not enough timeline data for distribution analysis.")
    print("Run A5_buildingeye_import.ipynb to import date data.")

## 7. Identify Stalled Projects

Projects with no activity for 180+ days that may need attention.

In [ ]:
# Identify stalled projects (>180 days inactive, not completed)
STALLED_THRESHOLD = CONFIG['timeline']['stalled_days_threshold']  # 180 days

if 'days_inactive' in df_timelines.columns:
    # Filter: inactive > threshold, not completed
    stalled_mask = (
        (df_timelines['days_inactive'] > STALLED_THRESHOLD) & 
        (df_timelines['is_completed'] != 1)
    )
    df_stalled = df_timelines[stalled_mask].sort_values('days_inactive', ascending=False)
    
    print(f"Stalled Projects (>{STALLED_THRESHOLD} days inactive):")
    print("="*60)
    print(f"Total: {len(df_stalled)} projects with {df_stalled['net_units'].sum():,.0f} units")
    
    if len(df_stalled) > 0:
        display_cols = ['address_display', 'net_units', 'status', 'first_filed_date', 
                       'last_status_date', 'days_inactive']
        available = [c for c in display_cols if c in df_stalled.columns]
        display(df_stalled[available].head(15))
else:
    print("Cannot identify stalled projects - no date data available.")

## 8. Completed Projects Analysis

Analyze projects that have received Certificate of Occupancy.

In [ ]:
# Completed projects
df_completed = df_timelines[df_timelines['is_completed'] == 1].copy()

print(f"Completed Projects:")
print("="*60)
print(f"Total: {len(df_completed)} projects with {df_completed['net_units'].sum():,.0f} units")

if len(df_completed) > 0 and 'days_total' in df_completed.columns:
    valid_total = df_completed['days_total'].dropna()
    if len(valid_total) > 0:
        print(f"\nTime to Completion:")
        print(f"  Average: {valid_total.mean():.0f} days ({valid_total.mean()/365:.1f} years)")
        print(f"  Median: {valid_total.median():.0f} days ({valid_total.median()/365:.1f} years)")
        print(f"  Fastest: {valid_total.min():.0f} days")
        print(f"  Slowest: {valid_total.max():.0f} days")
    
    # Show completed projects
    display_cols = ['address_display', 'net_units', 'first_filed_date', 'co_issued_date', 'days_total']
    available = [c for c in display_cols if c in df_completed.columns]
    print("\nRecently Completed:")
    display(df_completed.sort_values('co_issued_date', ascending=False)[available].head(10))
else:
    print("\nNo completion data available yet.")

## 9. Timeline by Project Size

Do larger projects take longer?

In [ ]:
# Analyze timeline by project size
if 'project_size_category' in df_timelines.columns and 'days_total' in df_timelines.columns:
    size_stats = df_timelines.groupby('project_size_category').agg({
        'days_total': ['count', 'mean', 'median'],
        'net_units': 'sum'
    }).round(0)
    
    size_stats.columns = ['Projects with Data', 'Avg Days', 'Median Days', 'Total Units']
    
    print("Timeline by Project Size:")
    display(size_stats)
else:
    print("Size category or timeline data not available.")

## 10. Permit Event Timeline

View recent permit activity across all projects.

In [ ]:
# Show recent permit events
if len(df_events) > 0:
    print("Recent Permit Events:")
    print("="*60)
    
    display_cols = ['event_date', 'address', 'permit_number', 'permit_type', 
                   'event_type', 'status', 'net_units']
    available = [c for c in display_cols if c in df_events.columns]
    
    display(df_events[available].head(20))
    
    # Event frequency by month
    if 'event_date' in df_events.columns:
        df_events['month'] = df_events['event_date'].dt.to_period('M')
        monthly = df_events.groupby('month').size()
        
        if len(monthly) > 3:
            plt.figure(figsize=(12, 4))
            monthly.plot(kind='bar', color='steelblue')
            plt.title('Permit Events by Month')
            plt.xlabel('Month')
            plt.ylabel('Number of Events')
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / 'events_by_month.png', dpi=150, bbox_inches='tight')
            plt.show()
else:
    print("No permit events loaded.")
    print("Run A5_buildingeye_import.ipynb to import event data.")

## 11. Export Timeline Data

In [ ]:
# Export timeline data to CSV
output_path = DATA_DIR / 'project_timelines.csv'
df_timelines.to_csv(output_path, index=False)
print(f"Saved timeline data: {output_path}")

# Export stalled projects
if 'df_stalled' in dir() and len(df_stalled) > 0:
    stalled_path = OUTPUT_DIR / 'stalled_projects.csv'
    df_stalled.to_csv(stalled_path, index=False)
    print(f"Saved stalled projects: {stalled_path}")

## 12. Summary Statistics

In [ ]:
# Generate summary report
print("\n" + "="*60)
print("BERKELEY HOUSING PIPELINE SUMMARY")
print("="*60)

print(f"\nTotal Projects: {len(df_timelines)}")
print(f"Total Units: {df_timelines['net_units'].sum():,.0f}")

# Date coverage
has_dates = df_timelines['first_filed_date'].notna().sum()
print(f"\nProjects with Timeline Data: {has_dates} ({100*has_dates/len(df_timelines):.1f}%)")

# Completion stats
completed = df_timelines['is_completed'].sum() if 'is_completed' in df_timelines.columns else 0
print(f"Completed Projects: {completed}")

# Stalled stats
if 'df_stalled' in dir():
    print(f"Stalled Projects (>{STALLED_THRESHOLD} days): {len(df_stalled)}")

# Average timeline
if 'days_total' in df_timelines.columns:
    avg_days = df_timelines['days_total'].mean()
    if pd.notna(avg_days):
        print(f"\nAverage Time to Completion: {avg_days:.0f} days ({avg_days/365:.1f} years)")

print("\n" + "="*60)

---

## Summary

This notebook:
- Loaded project data with timeline dates from the database
- Calculated duration metrics for each pipeline stage
- Classified projects by current pipeline stage
- Identified stalled projects needing attention
- Analyzed completed projects and completion times
- Visualized permit event activity

**Key Outputs:**
- `data/processed/project_timelines.csv` - Projects with calculated metrics
- `data/outputs/stalled_projects.csv` - Projects needing attention
- `data/outputs/pipeline_stages.png` - Pipeline visualization
- `data/outputs/timeline_distributions.png` - Duration histograms

**Next Steps:**
- Run `B2_status_classification.ipynb` for detailed status analysis
- Run `B3_progress_indicators.ipynb` for construction progress tracking
- View data in Datasette: https://berkeley-housing.fly.dev/